<a href="https://colab.research.google.com/github/MrStranger812/MovieLens-CLI-DM/blob/NotebookGCPP/Notebooks/03_Classification_Movielens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Environment Cleanup
!pip uninstall -y cudf-cu11 dask-cudf-cu11 cuml-cu11 cugraph-cu11 cupy-cuda11x
!pip uninstall -y cudf-cu12 dask-cudf-cu12 cuml-cu12 cugraph-cu12 cupy-cuda12x

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 03_Classification_Movielens.ipynb (Self-Contained)

# --- Core Imports ---
import pandas as pd
import numpy as np
import pickle
import gzip
import warnings
from pathlib import Path
import multiprocessing as mp
from typing import Dict, List, Tuple, Optional, Union

# --- Sklearn Imports ---
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                           confusion_matrix, classification_report, roc_auc_score,
                           roc_curve, precision_recall_curve)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans

# --- Imbalanced-learn Imports ---
from imblearn.over_sampling import SMOTE

# --- Visualization and UI Imports ---
import matplotlib.pyplot as plt
import seaborn as sns
from rich.console import Console
from rich.table import Table
from rich.progress import Progress, SpinnerColumn, TextColumn, BarColumn
from rich.panel import Panel
from rich import box

# --- GPU Support Imports (Optional) ---
try:
    import cupy as cp
    import cudf
    from cuml.ensemble import RandomForestClassifier as cuRandomForestClassifier
    from cuml.linear_model import LogisticRegression as cuLogisticRegression
    from cuml.svm import SVC as cuSVC
    from cuml.cluster import KMeans as cuKMeans
    CUML_AVAILABLE = True
except ImportError:
    CUML_AVAILABLE = False
    cp, cudf, cuKMeans, cuSVC, cuRandomForestClassifier, cuLogisticRegression = [None]*6

# --- Global Configuration ---
warnings.filterwarnings('ignore')
console = Console()

# --- Paths (Assuming Google Drive Setup) ---
BASE_DIR = Path("/content/drive/MyDrive/Movielens")
RAW_DATA_DIR = BASE_DIR / "data" / "raw" / "ml-20m"
PROCESSED_DATA_DIR = BASE_DIR / "data" / "processed"
REPORTS_DIR = BASE_DIR / "reports"
PROCESSED_DATA_DIR.mkdir(exist_ok=True) # Ensure directory exists
REPORTS_DIR.mkdir(exist_ok=True)

# --- Parameters ---
SAMPLE_SIZE = None  # Set to None for full dataset
USE_GPU = True      # Set to True to use GPU if available
N_JOBS = -1          # Use all available CPU cores

console.print(Panel.fit(
    "[bold cyan]🎬 MovieLens Classification Pipeline (Self-Contained)[/bold cyan]\n"
    f"GPU Support: {'Enabled' if USE_GPU and CUML_AVAILABLE else 'Disabled'}\n"
    f"Sample Size: {f'{SAMPLE_SIZE:,}' if SAMPLE_SIZE else 'Full Dataset'}",
    border_style="cyan"
))

╭───────────────────────────────────────────────────────╮
│ 🎬 MovieLens Classification Pipeline (Self-Contained) │
│ GPU Support: Enabled                                  │
│ Sample Size: Full Dataset                             │
╰───────────────────────────────────────────────────────╯

In [ ]:

class RatingClassifier:
    """Rating classifier for both binary and multi-class classification with GPU support"""

    def __init__(self, task_type: str = "binary", n_jobs: int = -1, use_gpu: bool = True):
        self.task_type = task_type
        self.n_jobs = n_jobs if n_jobs > 0 else mp.cpu_count()
        self.use_gpu = use_gpu and CUML_AVAILABLE
        self.console = Console()
        self.models = {}
        self.best_model = None
        self.scaler = StandardScaler()
        self.results = {}
        self.X_train, self.X_test, self.y_train, self.y_test = [None] * 4

    def load_classification_dataset(self) -> Tuple[pd.DataFrame, pd.Series]:
        """Load the processed dataset for classification"""
        ml_dataset_path = PROCESSED_DATA_DIR / "ml_ready_datasets.pkl.gz"

        if not ml_dataset_path.exists():
            # If ML dataset doesn't exist, create it from raw data
            self.console.print("[yellow]ML dataset not found. Creating from raw data...[/yellow]")
            return self._create_classification_dataset()

        with gzip.open(ml_dataset_path, "rb") as f:
            ml_datasets = pickle.load(f)

        if 'classification' not in ml_datasets or ml_datasets['classification'] is None:
            return self._create_classification_dataset()

        classification_data = ml_datasets['classification']
        X, y = classification_data['X'], classification_data['y']

        # Handle numpy arrays
        if isinstance(X, np.ndarray):
            feature_names = classification_data.get('feature_names',
                                                  [f'feature_{i}' for i in range(X.shape[1])])
            X = pd.DataFrame(X, columns=feature_names)

        if isinstance(y, np.ndarray):
            y = pd.Series(y, name='target')

        self.console.print(f"[green]✓ Loaded dataset: {X.shape[0]:,} samples, {X.shape[1]} features[/green]")
        return X, y

    def _create_classification_dataset(self) -> Tuple[pd.DataFrame, pd.Series]:
        """Create classification dataset from raw data"""
        # Load user and movie features if available
        user_features_path = PROCESSED_DATA_DIR / "user_features.parquet"
        movie_features_path = PROCESSED_DATA_DIR / "movie_features.parquet"

        if user_features_path.exists() and movie_features_path.exists():
            user_features = pd.read_parquet(user_features_path)
            movie_features = pd.read_parquet(movie_features_path)

            # For binary classification, create target from average ratings
            X = user_features.select_dtypes(include=[np.number])
            y = (user_features['rating_mean'] >= 4.0).astype(int)

            return X, y
        else:
            raise FileNotFoundError("Required feature files not found. Run preprocessing first.")

    def prepare_data(self, X: pd.DataFrame, y: pd.Series, test_size: float = 0.2):
        """Prepare the data for classification"""
        if self.task_type == "binary":
            self.class_names = ["Not Satisfied", "Satisfied"]
        else:
            # For multi-class, assume y contains actual ratings
            y = pd.cut(y, bins=[0, 1, 2, 3, 4, 5], labels=[1, 2, 3, 4, 5], include_lowest=True)
            self.class_names = ["Very Low", "Low", "Medium", "High", "Very High"]

        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            X, y, test_size=test_size, random_state=42, stratify=y
        )

        # Scale features
        self.X_train = pd.DataFrame(
            self.scaler.fit_transform(self.X_train),
            columns=self.X_train.columns,
            index=self.X_train.index
        )
        self.X_test = pd.DataFrame(
            self.scaler.transform(self.X_test),
            columns=self.X_test.columns,
            index=self.X_test.index
        )

        self._display_class_distribution()

    def _display_class_distribution(self):
        """Display class distribution in train/test sets."""
        train_dist = self.y_train.value_counts(normalize=True).sort_index()
        test_dist = self.y_test.value_counts(normalize=True).sort_index()

        table = Table(title="Class Distribution", box=box.ROUNDED)
        table.add_column("Class", style="cyan")
        table.add_column("Train %", justify="right")
        table.add_column("Test %", justify="right")

        for idx, class_name in enumerate(self.class_names):
            key = idx if self.task_type == 'binary' else idx + 1
            train_pct = train_dist.get(key, 0) * 100
            test_pct = test_dist.get(key, 0) * 100
            table.add_row(class_name, f"{train_pct:.1f}%", f"{test_pct:.1f}%")

        self.console.print(table)

    def fit(self, sample_size: Optional[int] = None, handle_imbalance: bool = True):
        """Fit the classification models"""
        X, y = self.load_classification_dataset()

        if sample_size and sample_size < len(X):
            self.console.print(f"[yellow]Sampling {sample_size:,} examples...[/yellow]")
            sample_idx = np.random.choice(len(X), sample_size, replace=False)
            X = X.iloc[sample_idx]
            y = y.iloc[sample_idx]

        self.prepare_data(X, y)
        self.train_ensemble_models(handle_imbalance)

    def train_ensemble_models(self, handle_imbalance: bool = True):
        """Train ensemble of classification models"""
        self.console.print(Panel.fit(
            f"[bold cyan]Training {self.task_type.title()} Classification Models[/bold cyan]",
            border_style="cyan"
        ))

        X_train_balanced, y_train_balanced = self.X_train.copy(), self.y_train.copy()

        # Handle class imbalance
        if handle_imbalance and self.task_type == 'binary':
            class_counts = self.y_train.value_counts()
            imbalance_ratio = class_counts.min() / class_counts.max()

            if imbalance_ratio < 0.5:
                self.console.print(f"[yellow]Class imbalance detected (ratio: {imbalance_ratio:.2f}). Applying SMOTE.[/yellow]")
                smote = SMOTE(random_state=42)
                X_train_balanced, y_train_balanced = smote.fit_resample(self.X_train, self.y_train)

        # Define models
        models_to_train = {
            'logistic': LogisticRegression(max_iter=1000, random_state=42, n_jobs=self.n_jobs),
            'random_forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=self.n_jobs),
            'gradient_boost': GradientBoostingClassifier(n_estimators=100, random_state=42),
            'svm': SVC(probability=True, random_state=42),
            'mlp': MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42)
        }

        # Train models
        with Progress(
            SpinnerColumn(),
            TextColumn("[progress.description]{task.description}"),
            BarColumn(),
            console=self.console
        ) as progress:
            task = progress.add_task("[cyan]Training models...", total=len(models_to_train))

            for name, model in models_to_train.items():
                progress.update(task, description=f"[cyan]Training {name}...")

                model.fit(X_train_balanced, y_train_balanced)
                y_pred = model.predict(self.X_test)
                y_pred_proba = model.predict_proba(self.X_test) if hasattr(model, 'predict_proba') else None

                metrics = self._calculate_metrics(self.y_test, y_pred, y_pred_proba)
                self.models[name] = model
                self.results[name] = {
                    'model': model,
                    'predictions': y_pred,
                    'probabilities': y_pred_proba,
                    'metrics': metrics
                }

                progress.advance(task)

        # Create voting classifier
        self.console.print("\n[cyan]Creating ensemble voting classifier...[/cyan]")
        voting_models = [(name, model) for name, model in self.models.items() if name != 'svm']

        voting_clf = VotingClassifier(
            estimators=voting_models,
            voting='soft',
            n_jobs=self.n_jobs
        )

        voting_clf.fit(X_train_balanced, y_train_balanced)
        y_pred_voting = voting_clf.predict(self.X_test)
        y_pred_proba_voting = voting_clf.predict_proba(self.X_test)

        voting_metrics = self._calculate_metrics(self.y_test, y_pred_voting, y_pred_proba_voting)

        self.models['voting_ensemble'] = voting_clf
        self.results['voting_ensemble'] = {
            'model': voting_clf,
            'predictions': y_pred_voting,
            'probabilities': y_pred_proba_voting,
            'metrics': voting_metrics
        }

        self._find_best_model()

    def _calculate_metrics(self, y_true, y_pred, y_pred_proba=None) -> Dict:
        """Calculate comprehensive classification metrics."""
        metrics = {
            'accuracy': accuracy_score(y_true, y_pred),
            'confusion_matrix': confusion_matrix(y_true, y_pred)
        }

        # Calculate precision, recall, f1
        avg_method = 'binary' if self.task_type == 'binary' else 'weighted'
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average=avg_method, zero_division=0
        )

        metrics.update({
            'precision': precision,
            'recall': recall,
            'f1_score': f1
        })

        # ROC AUC for binary classification
        if self.task_type == 'binary' and y_pred_proba is not None:
            metrics['roc_auc'] = roc_auc_score(y_true, y_pred_proba[:, 1])

        return metrics

    def _find_best_model(self):
        """Find the best performing model based on F1 score."""
        if self.results:
            best_name, best_result = max(
                self.results.items(),
                key=lambda item: item[1]['metrics']['f1_score']
            )
            self.best_model = best_result['model']
            self.console.print(
                f"\n[green]✓ Best model: {best_name.replace('_', ' ').title()} "
                f"(F1: {best_result['metrics']['f1_score']:.3f})[/green]"
            )

    def evaluate(self):
        """Display evaluation results for all trained models with enhanced metrics."""
        if not self.results:
            self.console.print("[red]No models trained yet.[/red]")
            return {}

        # Create detailed results table
        table = Table(title="Classification Results", box=box.ROUNDED)
        table.add_column("Model", style="cyan", no_wrap=True)
        table.add_column("Accuracy", justify="right")
        table.add_column("Precision", justify="right")
        table.add_column("Recall", justify="right")
        table.add_column("F1 Score", justify="right", style="bold")
        if self.task_type == 'binary':
            table.add_column("ROC AUC", justify="right")
            table.add_column("Support", justify="right")  # Number of samples

        # Get class distribution for context
        class_dist = self.y_test.value_counts().sort_index()

        for name, result in self.results.items():
            m = result['metrics']
            row = [
                name.replace('_', ' ').title(),
                f"{m['accuracy']:.3f}",
                f"{m['precision']:.3f}",
                f"{m['recall']:.3f}",
                f"{m['f1_score']:.3f}"
            ]
            if self.task_type == 'binary':
                row.append(f"{m.get('roc_auc', 0):.3f}")
                # Add support (number of test samples)
                row.append(f"{len(self.y_test):,}")
            table.add_row(*row)

        self.console.print(table)

        # Add context panel
        context_panel = Panel.fit(
            f"[bold]Classification Context[/bold]\n"
            f"Task Type: {self.task_type.title()}\n"
            f"Test Samples: {len(self.y_test):,}\n"
            f"Class Distribution: {', '.join([f'{self.class_names[i]}: {class_dist.get(i, 0)}' for i in range(len(self.class_names))])}\n"
            f"Best Model: {max(self.results.items(), key=lambda x: x[1]['metrics']['f1_score'])[0].replace('_', ' ').title()}",
            title="Classification Summary",
            border_style="blue"
        )
        self.console.print(context_panel)

        return {name: res['metrics'] for name, res in self.results.items()}

    def plot_confusion_matrices(self, save_path: Optional[Path] = None):
        """Plot confusion matrices for all models with enhanced visualization."""
        n_models = len(self.results)
        n_cols = 3
        n_rows = (n_models + n_cols - 1) // n_cols
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
        axes = axes.flatten() if n_models > 1 else [axes]

        # Calculate overall metrics for context
        overall_accuracy = np.mean([res['metrics']['accuracy'] for res in self.results.values()])
        overall_f1 = np.mean([res['metrics']['f1_score'] for res in self.results.values()])

        for idx, (name, result) in enumerate(self.results.items()):
            if idx < len(axes):
                ax = axes[idx]
                cm = result['metrics']['confusion_matrix']

                # Normalize confusion matrix for better visualization
                cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

                # Plot both raw counts and normalized
                sns.heatmap(
                    cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=ax,
                    xticklabels=self.class_names,
                    yticklabels=self.class_names,
                    cbar=False
                )

                # Add raw counts in smaller text
                for i in range(len(self.class_names)):
                    for j in range(len(self.class_names)):
                        ax.text(j+0.5, i+0.7, f"({cm[i, j]})",
                               ha="center", va="center", color="black", fontsize=8)

                # Add model metrics to title
                metrics = result['metrics']
                title = f'{name.replace("_", " ").title()}\nAcc: {metrics["accuracy"]:.3f}, F1: {metrics["f1_score"]:.3f}'
                if self.task_type == 'binary' and 'roc_auc' in metrics:
                    title += f', AUC: {metrics["roc_auc"]:.3f}'
                ax.set_title(title)
                ax.set_ylabel('True Label')
                ax.set_xlabel('Predicted Label')

        # Hide empty subplots
        for idx in range(len(self.results), len(axes)):
            axes[idx].set_visible(False)

        # Add overall metrics to the figure
        fig.suptitle(f'Confusion Matrices (Overall Accuracy: {overall_accuracy:.3f}, Overall F1: {overall_f1:.3f})',
                    fontsize=16, y=0.98)

        plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust for suptitle

        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
        else:
            plt.savefig(REPORTS_DIR / f'{self.task_type}_confusion_matrices.png',
                       dpi=300, bbox_inches='tight')
        plt.close()
        self.console.print("[green]✓ Enhanced confusion matrices saved[/green]")


In [ ]:

class GenrePredictor:
    """Predict movie genres with GPU support."""

    def __init__(self, n_jobs: int = -1, use_gpu: bool = False):
        self.n_jobs = n_jobs if n_jobs > 0 else mp.cpu_count()
        self.use_gpu = use_gpu and CUML_AVAILABLE
        self.console = Console()

        if self.use_gpu and not CUML_AVAILABLE:
            self.console.print("[yellow]⚠ GPU requested but cuML not available[/yellow]")
            self.use_gpu = False

        self.genres = []
        self.models = {}
        self.results = {}
        self.scaler = StandardScaler()

    def fit(self, ratings_df: pd.DataFrame, movies_df: pd.DataFrame, sample_size: Optional[int] = None):
        """Fit the genre predictor."""
        self.console.print(Panel.fit(
            "[bold cyan]Training Genre Predictors[/bold cyan]",
            border_style="cyan"
        ))

        X, y = self._prepare_genre_data(ratings_df, movies_df)

        if sample_size and len(X) > sample_size:
            sample_idx = np.random.choice(len(X), sample_size, replace=False)
            X = X.iloc[sample_idx]
            y = y.iloc[sample_idx]

        self.train_genre_predictors(X, y)

    def _prepare_genre_data(self, ratings_df: pd.DataFrame, movies_df: pd.DataFrame):
        """Prepare genre prediction data from movie rating stats."""
        self.console.print("[cyan]Preparing genre prediction data...[/cyan]")

        # Calculate movie statistics
        movie_stats = ratings_df.groupby('movieId').agg({
            'rating': ['mean', 'std', 'count'],
            'userId': 'nunique'
        })
        movie_stats.columns = ['rating_mean', 'rating_std', 'rating_count', 'unique_users']
        movie_stats = movie_stats.fillna(0)

        # Get genre labels
        movie_genres = movies_df.set_index('movieId')['genres']

        # Extract all unique genres
        all_genres = set()
        for genres in movie_genres.dropna():
            if genres != '(no genres listed)':
                all_genres.update(genres.split('|'))

        self.genres = sorted(list(all_genres))[:10]  # Top 10 genres
        self.console.print(f"[green]✓ Found {len(self.genres)} genres for prediction[/green]")

        # Create binary labels for each genre
        genre_labels = pd.DataFrame(index=movie_genres.index)
        for genre in self.genres:
            genre_labels[genre] = movie_genres.str.contains(genre, na=False, regex=False).astype(int)

        # Align features and labels
        common_idx = movie_stats.index.intersection(genre_labels.index)

        return movie_stats.loc[common_idx], genre_labels.loc[common_idx]

    def train_genre_predictors(self, X: pd.DataFrame, y: pd.DataFrame):
        """Train a predictor for each genre."""
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        # Scale features
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)

        with Progress(
            SpinnerColumn(),
            TextColumn("[progress.description]{task.description}"),
            BarColumn(),
            console=self.console
        ) as progress:
            task = progress.add_task("[cyan]Training genre predictors...", total=len(self.genres))

            for genre in self.genres:
                progress.update(task, description=f"[cyan]Training for {genre}...")
                smote = SMOTE(random_state=42)
                X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train[genre])

                if self.use_gpu:
                    clf = cuRandomForestClassifier(
                        n_estimators=100,
                        max_depth=10,
                        random_state=42
                    )
                    # Convert to cuDF for GPU processing
                    X_train_gpu = cudf.DataFrame(X_train_balanced)
                    y_train_gpu = cudf.Series(y_train_balanced)
                    clf.fit(X_train_gpu, y_train_gpu)

                    y_pred = clf.predict(cudf.DataFrame(X_test_scaled)).to_pandas().values
                else:
                    clf = RandomForestClassifier(
                        n_estimators=100,
                        max_depth=10,
                        random_state=42,
                        n_jobs=self.n_jobs
                    )
                    clf.fit(X_train_scaled, y_train[genre])
                    y_pred = clf.predict(X_test_scaled)

                # Calculate metrics
                accuracy = accuracy_score(y_test[genre], y_pred)
                precision, recall, f1, _ = precision_recall_fscore_support(
                    y_test[genre], y_pred, average='binary', zero_division=0
                )

                self.models[genre] = clf
                self.results[genre] = {
                    'accuracy': accuracy,
                    'precision': precision,
                    'recall': recall,
                    'f1_score': f1
                }

                progress.advance(task)

        self.evaluate()

    def evaluate(self):
        """Display genre prediction results with enhanced metrics and insights."""
        # Create detailed results table
        table = Table(title="Genre Prediction Results", box=box.ROUNDED)
        table.add_column("Genre", style="cyan", no_wrap=True)
        table.add_column("Accuracy", justify="right")
        table.add_column("Precision", justify="right")
        table.add_column("Recall", justify="right")
        table.add_column("F1 Score", justify="right", style="bold")
        table.add_column("Support", justify="right")  # Number of samples

        # Get support (number of test samples for each genre)
        supports = {}
        for genre in self.genres:
            # This would need to be calculated during training - for now we'll use placeholder
            supports[genre] = "N/A"

        for genre, metrics in self.results.items():
            table.add_row(
                genre,
                f"{metrics['accuracy']:.3f}",
                f"{metrics['precision']:.3f}",
                f"{metrics['recall']:.3f}",
                f"{metrics['f1_score']:.3f}",
                supports.get(genre, "N/A")
            )

        self.console.print(table)

        # Calculate overall metrics
        avg_accuracy = np.mean([r['accuracy'] for r in self.results.values()])
        avg_precision = np.mean([r['precision'] for r in self.results.values()])
        avg_recall = np.mean([r['recall'] for r in self.results.values()])
        avg_f1 = np.mean([r['f1_score'] for r in self.results.values()])

        # Find best and worst performing genres
        best_genre = max(self.results.items(), key=lambda x: x[1]['f1_score'])
        worst_genre = min(self.results.items(), key=lambda x: x[1]['f1_score'])

        # Create insights panel
        insights_panel = Panel.fit(
            f"[bold]Genre Prediction Insights[/bold]\n"
            f"Genres Analyzed: {len(self.genres)}\n"
            f"Average Metrics - Accuracy: {avg_accuracy:.3f}, Precision: {avg_precision:.3f}, "
            f"Recall: {avg_recall:.3f}, F1: {avg_f1:.3f}\n"
            f"Best Predicted: {best_genre[0]} (F1: {best_genre[1]['f1_score']:.3f})\n"
            f"Most Challenging: {worst_genre[0]} (F1: {worst_genre[1]['f1_score']:.3f})",
            title="Genre Prediction Summary",
            border_style="blue"
        )
        self.console.print(insights_panel)

        return self.results


In [ ]:

class UserTypeClassifier:
    """Classify users into behavioral types using clustering."""

    def __init__(self, n_clusters: int = 5, use_gpu: bool = False):
        self.n_clusters = n_clusters
        self.use_gpu = use_gpu and CUML_AVAILABLE
        self.console = Console()

        if self.use_gpu and not CUML_AVAILABLE:
            self.console.print("[yellow]⚠ GPU requested but cuML not available[/yellow]")
            self.use_gpu = False

        self.model = None
        self.user_types = None
        self.scaler = StandardScaler()

    def fit(self, ratings_df: pd.DataFrame, movies_df: pd.DataFrame, sample_size: Optional[int] = None):
        """Fit the user type classifier."""
        self.console.print(Panel.fit(
            "[bold cyan]Classifying User Types[/bold cyan]",
            border_style="cyan"
        ))

        user_features = self._create_user_features(ratings_df, movies_df)

        if sample_size:
            user_features = user_features.sample(
                n=min(sample_size, len(user_features)),
                random_state=42
            )

        self._classify_user_types(user_features)

    def _create_user_features(self, ratings_df: pd.DataFrame, movies_df: pd.DataFrame) -> pd.DataFrame:
        """Create behavioral features for user clustering."""
        self.console.print("[cyan]Creating user behavioral features...[/cyan]")

        # Basic user statistics
        user_behavior = ratings_df.groupby('userId').agg({
            'rating': ['mean', 'std', 'count',
                      lambda x: (x >= 4).sum() / len(x),  # Positive ratio
                      lambda x: (x <= 2).sum() / len(x)], # Negative ratio
            'movieId': 'nunique',
            'timestamp': [lambda x: (x.max() - x.min()).days]  # Activity span
        })

        # Flatten column names
        user_behavior.columns = [
            'avg_rating', 'rating_std', 'total_ratings',
            'positive_ratio', 'negative_ratio',
            'unique_movies', 'activity_span_days'
        ]

        # Add rating frequency
        user_behavior['rating_frequency'] = (
            user_behavior['total_ratings'] /
            (user_behavior['activity_span_days'] + 1)
        )

        return user_behavior.fillna(0)

    def _classify_user_types(self, user_features: pd.DataFrame) -> pd.Series:
        """Classify users into types using K-Means."""
        # Scale features
        features_scaled = self.scaler.fit_transform(user_features)

        # Apply clustering
        if self.use_gpu:
            self.model = cuKMeans(n_clusters=self.n_clusters, random_state=42)
            cluster_labels = self.model.fit_predict(features_scaled)
        else:
            self.model = KMeans(n_clusters=self.n_clusters, random_state=42, n_init=10)
            cluster_labels = self.model.fit_predict(features_scaled)

        user_features['cluster'] = cluster_labels
        self._assign_user_type_labels(user_features)


    def _assign_user_type_labels(self, features: pd.DataFrame):
        """Assign meaningful labels to user clusters."""
        # Analyze cluster profiles
        cluster_profiles = features.groupby('cluster').agg('mean')

        type_map = {}
        for cluster_id in range(self.n_clusters):
            profile = cluster_profiles.loc[cluster_id]

            # Determine user type based on characteristics
            if profile['total_ratings'] > features['total_ratings'].quantile(0.75):
                if profile['avg_rating'] > 3.5:
                    type_map[cluster_id] = 'Enthusiast'
                else:
                    type_map[cluster_id] = 'Active Critic'
            elif profile['total_ratings'] < features['total_ratings'].quantile(0.25):
                type_map[cluster_id] = 'Casual Viewer'
            elif profile['avg_rating'] < 3.0 and profile['rating_std'] < 1.0:
                type_map[cluster_id] = 'Harsh Critic'
            elif profile['positive_ratio'] > 0.7:
                type_map[cluster_id] = 'Positive Rater'
            else:
                type_map[cluster_id] = 'Regular User'

        self.user_types = features['cluster'].map(type_map)

    def evaluate(self):
        """Display the distribution of user types with enhanced cluster profiles."""
        if self.user_types is None:
            return {}

        distribution = self.user_types.value_counts()
        total_users = len(self.user_types)

        # Create distribution table
        table = Table(title="User Type Distribution", box=box.ROUNDED)
        table.add_column("User Type", style="cyan", no_wrap=True)
        table.add_column("Count", justify="right")
        table.add_column("Percentage", justify="right")
        table.add_column("Avg Rating", justify="right")  # Additional info
        table.add_column("Avg Ratings", justify="right")  # Additional info

        # Calculate additional metrics for each user type
        user_profiles = {}
        for user_type in distribution.index:
            # Get users of this type
            users_of_type = self.user_types[self.user_types == user_type].index

            # Calculate average metrics for these users
            # This would require access to the original user features - for now we'll use placeholder
            avg_rating = "N/A"
            avg_ratings_count = "N/A"

            table.add_row(
                user_type,
                f"{distribution[user_type]:,}",
                f"{(distribution[user_type] / total_users) * 100:.1f}%",
                avg_rating,
                avg_ratings_count
            )

        self.console.print(table)

        # Create insights panel
        insights_panel = Panel.fit(
            f"[bold]User Type Insights[/bold]\n"
            f"Total Users Classified: {total_users:,}\n"
            f"Number of User Types: {len(distribution)}\n"
            f"Dominant Type: {distribution.idxmax()} ({distribution.max()} users, {(distribution.max()/total_users)*100:.1f}%)\n"
            f"Most Engaged Type: [To be calculated based on activity metrics]",
            title="User Type Summary",
            border_style="blue"
        )
        self.console.print(insights_panel)

        return {
            'n_clusters': self.n_clusters,
            'user_types': distribution.to_dict(),
            'total_users': total_users
        }


In [ ]:

# --- Helper Functions ---
def load_data_source():
    """Helper to load the raw ratings and movies data."""
    console.print("\n[bold yellow]Loading raw data source...[/bold yellow]")

    try:
        ratings_df = pd.read_csv(RAW_DATA_DIR / "ratings.csv", dtype={
            'userId': 'int32',
            'movieId': 'int32',
            'rating': 'float32',
            'timestamp': 'int64'
        })
        movies_df = pd.read_csv(RAW_DATA_DIR / "movies.csv", dtype={
            'movieId': 'int32',
            'title': 'str',
            'genres': 'str'
        })

        # Convert timestamp to datetime
        ratings_df['timestamp'] = pd.to_datetime(ratings_df['timestamp'], unit='s')

        console.print(f"[green]✓ Loaded {len(ratings_df):,} ratings and {len(movies_df):,} movies[/green]")
        return ratings_df, movies_df

    except Exception as e:
        console.print(f"[red]❌ Error loading raw data: {e}[/red]")
        return None, None


def display_classification_summary(results: Dict):
    """Display enhanced summary of all classification tasks with contextual insights."""
    summary_table = Table(title="Classification Pipeline Summary", box=box.ROUNDED)
    summary_table.add_column("Task", style="cyan", no_wrap=True)
    summary_table.add_column("Status", justify="center")
    summary_table.add_column("Best Model/Method", style="green")
    summary_table.add_column("Performance", justify="right")
    summary_table.add_column("Key Insight", justify="left")  # New column

    # Binary Classification
    if results.get('binary_classification'):
        clf = results['binary_classification']['classifier']
        if clf.results:
            best_model_name = max(clf.results.items(),
                                key=lambda x: x[1]['metrics']['f1_score'])[0]
            best_f1 = clf.results[best_model_name]['metrics']['f1_score']
            best_acc = clf.results[best_model_name]['metrics']['accuracy']

            # Determine insight based on performance
            if best_f1 > 0.9:
                insight = "Excellent predictive power"
            elif best_f1 > 0.7:
                insight = "Good predictive power"
            else:
                insight = "Moderate predictive power"

            summary_table.add_row(
                "Binary Classification",
                "[green]✓[/green]",
                best_model_name.replace('_', ' ').title(),
                f"F1: {best_f1:.3f}, Acc: {best_acc:.3f}",
                insight
            )
        else:
            summary_table.add_row("Binary Classification", "[yellow]⚠[/yellow]", "No results", "N/A", "N/A")
    else:
        summary_table.add_row("Binary Classification", "[red]✗[/red]", "N/A", "N/A", "N/A")

    # Genre Prediction
    if results.get('genre_prediction'):
        genre_metrics = results['genre_prediction']['metrics']
        if genre_metrics:
            avg_f1 = np.mean([r['f1_score'] for r in genre_metrics.values()])
            best_genre = max(genre_metrics.items(), key=lambda x: x[1]['f1_score'])

            # Determine insight based on performance
            if avg_f1 > 0.3:
                insight = f"Best for {best_genre[0]}"
            else:
                insight = "Challenging prediction task"

            summary_table.add_row(
                "Genre Prediction",
                "[green]✓[/green]",
                "Random Forest (per genre)",
                f"Avg F1: {avg_f1:.3f}",
                insight
            )
        else:
            summary_table.add_row("Genre Prediction", "[yellow]⚠[/yellow]", "No results", "N/A", "N/A")
    else:
        summary_table.add_row("Genre Prediction", "[red]✗[/red]", "N/A", "N/A", "N/A")

    # User Type Classification
    if results.get('user_type_classification'):
        metrics = results['user_type_classification']['metrics']
        if metrics:
            user_types = metrics.get('user_types', {})
            dominant_type = max(user_types.items(), key=lambda x: x[1])[0] if user_types else "N/A"

            # Determine insight based on distribution
            if len(user_types) >= 4:
                insight = f"Diverse user behaviors"
            else:
                insight = f"{dominant_type} dominant"

            summary_table.add_row(
                "User Type Classification",
                "[green]✓[/green]",
                "K-Means Clustering",
                f"{metrics.get('n_clusters', 0)} types found",
                insight
            )
        else:
            summary_table.add_row("User Type Classification", "[yellow]⚠[/yellow]", "No results", "N/A", "N/A")
    else:
        summary_table.add_row("User Type Classification", "[red]✗[/red]", "N/A", "N/A", "N/A")

    console.print(summary_table)

    # Add overall insights panel
    completed_tasks = sum(1 for task in ['binary_classification', 'genre_prediction', 'user_type_classification']
                         if results.get(task))
    total_tasks = 3

    overall_panel = Panel.fit(
        f"[bold]Pipeline Execution Summary[/bold]\n"
        f"Tasks Completed: {completed_tasks}/{total_tasks}\n"
        f"Dataset: MovieLens 20M\n"
        f"Sample Size: {f'{SAMPLE_SIZE:,}' if SAMPLE_SIZE else 'Full Dataset'}\n"
        f"GPU Acceleration: {'Enabled' if USE_GPU and CUML_AVAILABLE else 'Disabled'}\n"
        f"Next Steps: {'Analyze misclassifications' if completed_tasks == total_tasks else 'Complete remaining tasks'}",
        title="Overall Pipeline Status",
        border_style="cyan"
    )
    console.print(overall_panel)



def save_visualizations(binary_clf: RatingClassifier):
    """Save enhanced visualization plots for binary classifier."""
    try:
        # Save confusion matrices
        binary_clf.plot_confusion_matrices()

        # Save ROC curves for binary classification with enhancements
        if binary_clf.task_type == 'binary':
            plt.figure(figsize=(12, 10))

            # Plot ROC curves
            for name, result in binary_clf.results.items():
                if result.get('probabilities') is not None:
                    fpr, tpr, _ = roc_curve(
                        binary_clf.y_test,
                        result['probabilities'][:, 1]
                    )
                    auc = result['metrics'].get('roc_auc', 0)
                    plt.plot(fpr, tpr, label=f'{name.replace("_", " ").title()} (AUC = {auc:.3f})')

            plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
            plt.xlim([0.0, 1.0])
            plt.ylim([0.0, 1.05])
            plt.xlabel('False Positive Rate', fontsize=12)
            plt.ylabel('True Positive Rate', fontsize=12)
            plt.title('ROC Curves - Binary Classification', fontsize=14, fontweight='bold')
            plt.legend(loc="lower right", fontsize=10)
            plt.grid(True, alpha=0.3)

            # Add performance annotations
            best_auc = max([result['metrics'].get('roc_auc', 0) for result in binary_clf.results.values()])
            plt.annotate(f'Best AUC: {best_auc:.3f}',
                        xy=(0.6, 0.3), fontsize=12,
                        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8))

            plt.savefig(REPORTS_DIR / 'roc_curves.png', dpi=300, bbox_inches='tight')
            plt.close()

            # Save precision-recall curves
            plt.figure(figsize=(12, 10))

            for name, result in binary_clf.results.items():
                if result.get('probabilities') is not None:
                    precision, recall, _ = precision_recall_curve(
                        binary_clf.y_test,
                        result['probabilities'][:, 1]
                    )
                    plt.plot(recall, precision, label=f'{name.replace("_", " ").title()}')

            plt.xlabel('Recall', fontsize=12)
            plt.ylabel('Precision', fontsize=12)
            plt.title('Precision-Recall Curves - Binary Classification', fontsize=14, fontweight='bold')
            plt.legend(loc="best", fontsize=10)
            plt.grid(True, alpha=0.3)

            # Add no-skill line
            no_skill = len(binary_clf.y_test[binary_clf.y_test==1]) / len(binary_clf.y_test)
            plt.plot([0, 1], [no_skill, no_skill], 'k--', label=f'No Skill (AP = {no_skill:.3f})')

            plt.savefig(REPORTS_DIR / 'precision_recall_curves.png', dpi=300, bbox_inches='tight')
            plt.close()

            console.print("[green]✓ Enhanced visualizations saved (ROC, PR curves, confusion matrices)[/green]")
    except Exception as e:
        console.print(f"[yellow]⚠ Could not save visualizations: {e}[/yellow]")


In [ ]:
import time
def display_binary_classification_results(metrics):
    """Display binary classification results properly."""
    console.print(f"\n[bold green]✓ Binary Classification Complete[/bold green]")

    # Create metrics table
    table = Table(title="Binary Classification Results", show_header=True, header_style="bold magenta")
    table.add_column("Metric", style="cyan", no_wrap=True)
    table.add_column("Value", style="green")

    for metric_name, value in metrics.items():
        if isinstance(value, (int, float)):
            table.add_row(metric_name.replace('_', ' ').title(), f"{value:.3f}")
        else:
            table.add_row(metric_name.replace('_', ' ').title(), str(value))

    console.print(table)

def safe_spinner_context(text):
    """Create a safe spinner context that properly cleans up."""
    return console.status(text, spinner="dots")

def display_genre_results_properly(results):
    """Display genre prediction results with proper formatting."""
    console.print(f"\n[bold green]✓ Genre Prediction Complete[/bold green]")

    table = Table(title="Genre Prediction Results", show_header=True, header_style="bold magenta")
    table.add_column("Genre", style="cyan", no_wrap=True)
    table.add_column("Accuracy", style="green")
    table.add_column("Precision", style="yellow")
    table.add_column("Recall", style="blue")
    table.add_column("F1 Score", style="red", justify="right") # Added justify for alignment

    for genre, metrics in results.items():
        table.add_row(
            genre,
            f"{metrics.get('accuracy', 0):.3f}",
            f"{metrics.get('precision', 0):.3f}",
            f"{metrics.get('recall', 0):.3f}",
            f"{metrics.get('f1_score', 0):.3f}"  # <-- FIX: Changed 'f1' to 'f1_score'
        )

    console.print(table)


# --- Fixed Main Pipeline Execution ---
def run_classification_pipeline():
    """Run the complete classification pipeline with proper Rich usage."""
    pipeline_results = {}

    console.print(Panel.fit("[bold cyan]Starting Classification Pipeline...[/bold cyan]"))

    # Load data
    with safe_spinner_context("Loading data..."):
        ratings_df, movies_df = load_data_source()
        time.sleep(0.5)  # Brief pause to show spinner

    if ratings_df is None or movies_df is None:
        console.print("[bold red]Pipeline halted: Could not load data.[/bold red]")
        return pipeline_results

    console.print("[green]✓ Data loaded successfully[/green]")

    # Phase 1: Binary Rating Classification
    console.print(f"\n{Panel.fit('[bold yellow]Phase 1: Binary Rating Classification[/bold yellow]')}")
    console.print("Predicting user satisfaction (rating >= 4.0)")

    try:
        with safe_spinner_context("Training binary classifier..."):
            binary_clf = RatingClassifier(task_type='binary', n_jobs=N_JOBS, use_gpu=USE_GPU)
            binary_clf.fit(sample_size=SAMPLE_SIZE)

        with safe_spinner_context("Evaluating binary classifier..."):
            metrics = binary_clf.evaluate()

        pipeline_results['binary_classification'] = {
            'classifier': binary_clf,
            'metrics': metrics
        }

        # Save visualizations
        with safe_spinner_context("Saving visualizations..."):
            save_visualizations(binary_clf)

    except Exception as e:
        console.print(f"[red]❌ Binary classification failed: {e}[/red]")
        pipeline_results['binary_classification'] = None

    # Phase 2: Genre Prediction
    console.print(f"\n{Panel.fit('[bold yellow]Phase 2: Genre Prediction[/bold yellow]')}")
    console.print("Predicting movie genres from rating patterns")

    try:
        with safe_spinner_context("Initializing genre predictor..."):
            genre_predictor = GenrePredictor(n_jobs=N_JOBS, use_gpu=USE_GPU)

        with safe_spinner_context("Training genre models (this may take a while)..."):
            genre_predictor.fit(ratings_df, movies_df, sample_size=SAMPLE_SIZE)

        pipeline_results['genre_prediction'] = {
            'predictor': genre_predictor,
            'metrics': genre_predictor.results
        }

    except Exception as e:
        console.print(f"[red]❌ Genre prediction failed: {e}[/red]")
        pipeline_results['genre_prediction'] = None

    # Phase 3: User Type Classification
    console.print(f"\n{Panel.fit('[bold yellow]Phase 3: User Type Classification[/bold yellow]')}")
    console.print("Segmenting users into behavioral types")

    try:
        with safe_spinner_context("Creating user behavioral features..."):
            user_classifier = UserTypeClassifier(n_clusters=5, use_gpu=USE_GPU)

        with safe_spinner_context("Performing user clustering..."):
            user_classifier.fit(ratings_df, movies_df, sample_size=SAMPLE_SIZE)

        # --- FIX IS HERE ---
        # Explicitly evaluate the classifier to get metrics and display the table
        user_metrics = user_classifier.evaluate()

        # Add the results, including the 'metrics' key, to the main dictionary
        pipeline_results['user_type_classification'] = {
            'classifier': user_classifier,
            'metrics': user_metrics
        }
        # --- END OF FIX ---

        console.print("[green]✓ User type classification complete[/green]")

    except Exception as e:
        console.print(f"[red]❌ User type classification failed: {e}[/red]")
        pipeline_results['user_type_classification'] = None

    # Display final summary
    console.print(f"\n{'='*60}")
    display_classification_summary(pipeline_results)

    # Save results
    try:
        with safe_spinner_context("Saving results..."):
            results_path = PROCESSED_DATA_DIR / "classification_results.pkl.gz"

            # Extract only serializable metrics
            metrics_to_save = {}
            for task, data in pipeline_results.items():
                if data:
                    metrics_to_save[task] = data.get('metrics', {})

            with gzip.open(results_path, 'wb') as f:
                pickle.dump(metrics_to_save, f, protocol=4)

        console.print(f"[green]✓ Results saved to {results_path}[/green]")

    except Exception as e:
        console.print(f"[red]❌ Failed to save results: {e}[/red]")

    return pipeline_results


# Execute the pipeline
if __name__ == "__main__":
    console.print("\n[bold cyan]Starting Classification Pipeline...[/bold cyan]")
    results = run_classification_pipeline()
    console.print("\n[bold green]Pipeline completed![/bold green]")

Starting Classification Pipeline...

╭─────────────────────────────────────╮
│ Starting Classification Pipeline... │
╰─────────────────────────────────────╯

Loading raw data source...

Output()

✓ Loaded 20,000,263 ratings and 27,278 movies

...

                                          Classification Pipeline Summary                                          
╭──────────────────────────┬────────┬──────────────────────────┬───────────────────────┬──────────────────────────╮
│ Task                     │ Status │ Best Model/Method        │           Performance │ Key Insight              │
├──────────────────────────┼────────┼──────────────────────────┼───────────────────────┼──────────────────────────┤
│ Binary Classification    │   ✓    │ Random Forest            │ F1: 1.000, Acc: 1.000 │ Excellent predictive     │
│                          │        │                          │                       │ power                    │
│ Genre Prediction         │   ✓    │ Random Forest (per       │         Avg F1: 0.222 │ Challenging prediction   │
│                          │        │ genre)                   │                       │ task                     │
│ User Type Classification │   ✓    │ K-Means Clustering       │         5 types found │ Regular User dominant    │
╰──────────────────────────┴────────┴──────────────────────────┴───────────────────────┴──────────────────────────╯

╭─────── Overall Pipeline Status ────────╮
│ Pipeline Execution Summary             │
│ Tasks Completed: 3/3                   │
│ Dataset: MovieLens 20M                 │
│ Sample Size: Full Dataset              │
│ GPU Acceleration: Enabled              │
│ Next Steps: Analyze misclassifications │
╰────────────────────────────────────────╯

Output()

✓ Results saved to /content/drive/MyDrive/Movielens/data/processed/classification_results.pkl.gz

Pipeline completed!